# **Transformer**

<p align="center">
  <img src="Transformer Arch.png" width="300" alt="Transformer Architecture">
  <br>
  This is the architecture of a Transformer 
</p>

---

In [1]:
import torch
import torch.nn as nn
import math
from transformers import AutoTokenizer      # Only for Tokenization

---

In [2]:
class Tokenisation:
    def __init__(self, model_name="gpt2"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    @property
    def vocab_size(self):
        return self.tokenizer.vocab_size

    def encode_batch(self, texts):
        return self.tokenizer(
            texts,
            truncation=False,
            return_attention_mask=False
        )

In [3]:
class InputPositionalEmbeddings(nn.Module):
    def __init__(self, vocab_size:int, d_model:int, max_seq_len:int=512):
        super().__init__()
        self.token_embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        self.position_embedding = nn.Embedding(num_embeddings=max_seq_len, embedding_dim=d_model)

    def forward(self, x:torch.Tensor):
        # x shape: (batch_size, sequence_length)
        token_emb = self.token_embedding(x)

        # Create position IDs: 0, 1, 2, ..., sequence_length - 1
        positions = torch.arange(
            x.size(1),
            device=x.device
        )

        # position_emb shape: (sequence_length, d_model)
        position_emb = self.position_embedding(positions)

        return token_emb + position_emb

Input dimensions:
$$
\mathbf{X} \in \mathbb{R}^{\text{Batch\_size}\ \times\ \text{seq\_len}\ \times\ d_{\text{model}}}
$$

X.shape = (Batch_size, seq_len, d_model)

where:
- **Batch_size**: Number of sentences/sequences processed in parallel in a batch.
- **seq_len**: Number of tokens (words or subwords) per sequence.
- **d_model**: Dimensions of the Embedding

### Attention mechanism

**Linear Transformation (Create Q, K, V)**

Three learned weight matrices: 
$$
W_Q, W_K, W_V \in \mathbb{R}^{d_{model}\ \times\ d_k} \\
Q, K, V \in \mathbb{R}^{seq\_len\ \times\ d_k}
$$

$$
Q = XW_Q \\
K = XW_K \\
V = XW_V \\
$$

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, h:int, d_model:int, d_k:int):
        super().__init__()
        self.h = h
        assert h * d_k == d_model, f"Embedding dimension d_model ({d_model}) must equal num_heads ({h}) * head_dim ({d_k})"

        self.d_k = d_k 
        self.d_model = d_model

        self.W_Q = nn.Parameter(torch.randn((h, d_model, d_k)))
        self.W_K = nn.Parameter(torch.randn((h, d_model, d_k)))
        self.W_V = nn.Parameter(torch.randn((h, d_model, d_k)))


        self.W_O = nn.Parameter(torch.randn((d_model, d_model)))

    def forward(self, X:torch.Tensor):
        outputs = []
        attentions = []

        seq_len = X.shape[1]

        mask = torch.triu(torch.ones(seq_len, seq_len, device=X.device), diagonal=1).bool()

        for head in range(self.h):
            Q = X @ self.W_Q[head]
            K = X @ self.W_K[head]
            V = X @ self.W_V[head]


            scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)    # (Q.K)/sqrt(d_k)

            scores = scores.masked_fill(mask=mask, value=float('-inf'))

            attention = torch.softmax(scores, dim=-1)
            attentions.append(attention)

            head_output = attention @ V     # head_i = attention * V

            outputs.append(head_output)

        outputs = torch.cat(outputs, dim=-1)
        outputs = outputs @ self.W_O

        return outputs, attentions

In [5]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff=1024):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, X:torch.Tensor):
        return self.linear2(torch.relu(self.linear1(X)))

In [6]:
class LayerNormalization(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()

        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.d_model = d_model

    def forward(self, X: torch.Tensor):
        mean = X.mean(dim=-1, keepdim=True)
        var = ((X - mean) ** 2).mean(dim=-1, keepdim=True)

        normalized = (X - mean) / torch.sqrt(var + self.eps)

        return self.alpha * normalized + self.beta

In [7]:
class ResidualConnection(nn.Module):        # Y = LayerNorm(X + Attention(X))   ----> ADD & NORM
    def __init__(self, d_model:int):
        super().__init__()
        self.norm = LayerNormalization(d_model)

    def forward(self, X:torch.Tensor, sublayer:torch.Tensor):
        return self.norm(X + sublayer)

In [8]:
class SingleTransformerBlock(nn.Module):
    def __init__(self, d_model:int, d_k, h:int):
        super().__init__()
        self.MHA = MultiHeadAttention(h, d_model, d_k)
        self.ResidualConnection = ResidualConnection(d_model)
        self.FFN = FeedForwardNetwork(d_model, d_ff=1024)

    def forward(self, X:torch.Tensor):
        outputs, attentions = self.MHA(X)     # Multihead Attention --> outputs, attentions
        X = self.ResidualConnection(X, outputs)     # Add & Norm
        ffn_output = self.FFN(X)     # Feed Forward Network
        X = self.ResidualConnection(X, ffn_output)      # Add & Norm

        return X, attentions
        

In [9]:
class OutputLayer(nn.Module):
    def __init__(self, d_model:int, vocab_size:int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size

        self.norm = LayerNormalization(d_model)
        self.linear = nn.Linear(self.d_model, self.vocab_size)

    def forward(self, X:torch.Tensor):
        X = self.norm(X)
        logits = self.linear(X)
        return logits

**Transformer Model**

In [10]:
class Transformer(nn.Module):
    def __init__(self, d_model:int, d_k, h:int=8):
        super().__init__()

        # Tokeniser
        self.tokeniser = Tokenisation()
        self.vocab_size = self.tokeniser.vocab_size

        # Embedding
        self.InputPositionalEmbedding = InputPositionalEmbeddings(self.vocab_size, d_model)

        # Transformer Blocks
        self.blocks = nn.ModuleList([
            SingleTransformerBlock(d_model, d_k, h),
            SingleTransformerBlock(d_model, d_k, h),
            SingleTransformerBlock(d_model, d_k, h),
            SingleTransformerBlock(d_model, d_k, h)
        ])

        # Output Layer
        self.output = OutputLayer(d_model, self.vocab_size)

    def forward(self, X:torch.Tensor):
        X = self.InputPositionalEmbedding(X)        # Input + Positional embedding

        all_attentions = []

        for block in self.blocks:
            X, attentions = block(X)
            all_attentions.append(attentions)

        return self.output(X), all_attentions

    def tokenise(self, sentences, max_length=20):
        tokens = self.tokeniser.encode_batch(
            sentences,
        )
        return tokens["input_ids"]

    @torch.no_grad()
    def generate(model, prompt, max_new_tokens=50, temperature=1.0, top_k=50):
        device = next(model.parameters()).device

        # Tokenise prompt
        tokens = model.tokenise([prompt])[0]
        x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

        model.eval()

        for _ in range(max_new_tokens):

            # Keep only the most recent seq_len tokens if necessary
            x_input = x[:, -128:]

            # Forward pass
            logits, _ = model(x_input)

            # We only care about the LAST token position
            next_token_logits = logits[:, -1, :]

            # Temperature
            next_token_logits = next_token_logits / temperature

            # Top-k sampling
            if top_k is not None:
                values, indices = torch.topk(next_token_logits, top_k)
                
                filtered_logits = torch.full_like(
                    next_token_logits,
                    float("-inf")
                )

                filtered_logits.scatter_(1, indices, values)

                next_token_logits = filtered_logits

            # Convert logits → probabilities
            probabilities = torch.softmax(next_token_logits, dim=-1)

            # Sample one token
            next_token = torch.multinomial(probabilities, num_samples=1)

            # Append it
            x = torch.cat([x, next_token], dim=1)

        # Convert token IDs back to text
        generated_tokens = x[0].tolist()

        text = model.tokeniser.tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )

        return text

---

### Training from the **TinyStories** Dataset from `HuggingFace`

In [11]:
tokeniser = Tokenisation()

In [ ]:
from datasets import load_from_disk

# Point it to the folder containing the .arrow files
ds = load_from_disk("./data/tinystories")

small_ds = ds["train"].select(range(100000))

def tokenize_function(batch):
    return tokeniser.tokenizer(
        batch["text"],
        truncation=False,
        return_attention_mask=False
    )

tokenized_ds = small_ds.map(
    tokenize_function,
    batched=True,
    batch_size=32,
    remove_columns=["text"]
)

In [31]:
from torch.utils.data import Dataset

class TinyStoriesDataset(Dataset):

    def __init__(self, tokenized_ds, seq_len):
        self.seq_len = seq_len
        self.examples = []

        for example in tokenized_ds:
            tokens = example["input_ids"]

            # Create chunks from this story
            for i in range(0, len(tokens) - seq_len, seq_len):
                chunk = tokens[i:i + seq_len + 1]

                if len(chunk) == seq_len + 1:
                    self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        chunk = self.examples[idx]

        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)

        return x, y

In [32]:
train_dataset = TinyStoriesDataset(tokenized_ds, seq_len=150)
X, Y = train_dataset[0]

In [33]:
from torch.utils.data import DataLoader


def train(model: Transformer, train_dataset: TinyStoriesDataset, epochs: int = 3, batch_size: int = 16, lr:float=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,   # keeps every batch the same shape
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0.0

        for batch, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            logits, _ = model(x)              # (batch, seq_len, vocab_size)

            # flatten for CrossEntropyLoss: it wants (N, C) vs (N,)
            loss = criterion(
                logits.view(-1, logits.size(-1)),   # (batch*seq_len, vocab_size)
                y.view(-1)                          # (batch*seq_len,)
            )

            loss.backward()
            # torch.nn.utils.clip_grad_norm_(model.parameters())
            optimizer.step()

            total_loss += loss.item()

            print(f"Epoch {epoch+1} | Batch {batch} | Loss {loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        print(f"== Epoch {epoch+1} done | Avg loss {avg_loss:.4f} | PPL {math.exp(avg_loss):.2f} ==")

    return model

In [ ]:
model = Transformer(d_model=240, d_k=60, h=4)
model = train(model, train_dataset, epochs=4, batch_size=30, lr=5e-4)

In [36]:
model.generate("Once upon a ", max_new_tokens=10)

'Once upon a  upon a was He little loved a a. to'